In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
(
  SELECT *
  FROM (
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
  )
  WHERE FILL_DATE BETWEEN '2023-04-01' AND '2025-03-31'
);


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Specified AS
(
  SELECT *
  FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service,
      PROCEDURE_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service,
      NULL AS PROCEDURE_CODE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31'
);

In [0]:

CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Specified AS
SELECT
  PATIENT_ID
FROM MPSII_1Dx_Specified
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;
select count(distinct patient_id) from MPSII_2Dx_Specified

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_1Dx_Unspecified AS
(
  SELECT *
  FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31'
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_2Dx_Unspecified
AS 
(SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM MPSII_1Dx_Unspecified--TABLE
--WHERE ARRAYS_OVERLAP (SPLIT(DIAGNOSIS_CODES, '|'), ARRAY_CONSTRUCT_COMPACT('E761'))
--WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31' 
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2);

In [0]:
Create or replace temp view MPSII_2Dx_Tx_Specified_Tx_claims AS
Select * from MPSII_TREATMENT_TABLE
where patient_id in (select distinct patient_id from MPSII_2Dx_Specified);

In [0]:
create or replace temp view MPSII_Incremental_Patients as
Select distinct patient_id from MPSII_2Dx_Unspecified
where patient_id in (
Select distinct patient_id from MPSII_TREATMENT_TABLE
where code in ('54092070001','540920700','J1743')
)
AND patient_id not in (select distinct patient_id from MPSII_2Dx_Tx_Specified_Tx_claims);

In [0]:
create or replace temp view MPSII_Specified_And_Incremental as
select * from MPSII_1Dx_Specified where patient_id in (select distinct patient_id from MPSII_2Dx_Specified)
union
select *, NULL AS PROCEDURE_CODE from MPSII_1Dx_Unspecified where patient_id in (select distinct patient_id from MPSII_Incremental_Patients) 


In [0]:
create or replace temp view MPSII_All_Dx_Elaprase_Treated as
select * from mpsII_treatment_table
where patient_id in (select distinct patient_id from MPSII_Specified_And_Incremental)

In [0]:
select * from MPSII_2Dx_Tx_Specified_Tx_claims_2023_to_25
union
Select * from elaprase 
where patient_id in (select distinct patient_id from MPSII_Incremental_Patients)
and code in ('54092070001','540920700','J1743')

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW MPSII_all_Dx_1167 AS
SELECT a.*,
       b.HCO_PRIMARY_NPI,
       b.PRIMARY_SPECIALTY,
       b.SECONDARY_SPECIALTY,
       CASE 
         WHEN primary_specialty LIKE '%Genetic%' OR secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
         WHEN primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
         WHEN primary_specialty LIKE '%Psychiatry & Neurology%' OR secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
              primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
         WHEN primary_specialty LIKE '%Nurse Practitioner%' OR primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
         WHEN primary_specialty LIKE '%Internal Medicine%' OR secondary_specialty LIKE '%Internal Medicine%' THEN 'PCP'
         WHEN primary_specialty LIKE '%Family Medicine%' OR secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
         WHEN a.npi IS NULL THEN 'NA'
         ELSE 'Others'
       END AS SPECIALTY
FROM (
  SELECT * FROM MPSII_1Dx_Specified
  WHERE patient_id IN (SELECT DISTINCT patient_id FROM MPSII_2Dx_Specified)

  UNION

  SELECT *, null as procedure_codes FROM MPSII_1Dx_Unspecified
  WHERE patient_id IN (SELECT DISTINCT patient_id FROM MPSII_Incremental_Patients)
) a
LEFT JOIN com_edp_prd.com_raw.kom_providers b
  ON a.npi = b.npi
;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW Dx_1_1_Mapped_HCP_Refresh AS
SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID,
    Fill_date,
    'DX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date, 
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM MPSII_all_Dx_1167
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
);

In [0]:
create or replace temp view MPSII_all_Tx_2023_to_25_425 as
select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from MPSII_2Dx_Tx_Specified_Tx_claims
union
Select * from MPSII_TREATMENT_TABLE
where patient_id in (select distinct patient_id from MPSII_Incremental_Patients)
and code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi;

In [0]:
select distinct * from mpsii_treatment_table where patient_id in (select distinct patient_id from mpsii_incremental_patients)

In [0]:
Create or replace temp view MPSII_all_DX_TX AS
SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID,
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM MPSII_all_Tx_2023_to_25_425
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
);

In [0]:
create or replace temp view MPSII_Elaprase_Primary_Tag as
select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
      select * from MPSII_Treatment_table
    ) a
    left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi;

In [0]:
Create or replace temp view MPSII_elaprase_treated_npi_tag AS
SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM MPSII_Elaprase_Primary_Tag
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
);

In [0]:
create or replace temp view HCP_Universe as
select * from 
(select distinct npi from Dx_1_1_Mapped_HCP_Refresh
union
select distinct npi from MPSII_all_DX_TX
union 
select distinct npi from MPSII_elaprase_treated_npi_tag)

In [0]:
-- Unified DX + TX view
CREATE OR REPLACE TEMP VIEW MPSII_DX_TX_UNION AS

-- DX branch (add NULLs for TX-only fields)
SELECT
  d.NPI,
  d.SPECIALTY,
  d.PATIENT_ID,
  d.FILL_DATE,
  CAST(NULL AS STRING) AS KH_PLAN,
  CAST(NULL AS STRING) AS HCO_PRIMARY_NPI,
  CAST(NULL AS STRING) AS PLACE_OF_SERVICE,
  'DX' AS PATIENT_TYPE
FROM Dx_1_1_Mapped_HCP_Refresh d

UNION

-- TX branch 1
SELECT
  t.NPI,
  t.SPECIALTY,
  t.PATIENT_ID,
  t.FILL_DATE,
  CAST(t.KH_PLAN AS STRING)         AS KH_PLAN,
  CAST(t.HCO_PRIMARY_NPI AS STRING) AS HCO_PRIMARY_NPI,
  CAST(t.PLACE_OF_SERVICE AS STRING)AS PLACE_OF_SERVICE,
  'TX' AS PATIENT_TYPE
FROM MPSII_all_DX_TX t

UNION

-- TX branch 2
SELECT
  e.NPI,
  e.SPECIALTY,
  e.PATIENT_ID,
  e.FILL_DATE,
  CAST(e.KH_PLAN AS STRING)         AS KH_PLAN,
  CAST(e.HCO_PRIMARY_NPI AS STRING) AS HCO_PRIMARY_NPI,
  CAST(e.PLACE_OF_SERVICE AS STRING)AS PLACE_OF_SERVICE,
  'TX' AS PATIENT_TYPE
FROM MPSII_elaprase_treated_npi_tag e;


In [0]:
CREATE OR REPLACE TEMP VIEW metric_numbers AS
WITH dx AS (
  SELECT npi, COUNT(DISTINCT patient_id) AS pts_dx
  FROM Dx_1_1_Mapped_HCP_Refresh
  WHERE npi IS NOT NULL AND patient_id IS NOT NULL
  GROUP BY npi
),
tx1 AS (
  SELECT npi, COUNT(DISTINCT patient_id) AS pts_all_dx_tx
  FROM MPSII_all_DX_TX
  WHERE npi IS NOT NULL AND patient_id IS NOT NULL
  GROUP BY npi
),
tx2 AS (
  SELECT npi, COUNT(DISTINCT patient_id) AS pts_elaprase_treated
  FROM MPSII_elaprase_treated_npi_tag
  WHERE npi IS NOT NULL AND patient_id IS NOT NULL
  GROUP BY npi
),
tot AS (
  SELECT npi, COUNT(DISTINCT patient_id) AS pts_unique_all
  FROM (
    SELECT npi, patient_id FROM Dx_1_1_Mapped_HCP_Refresh
    UNION
    SELECT npi, patient_id FROM MPSII_all_DX_TX
    UNION
    SELECT npi, patient_id FROM MPSII_elaprase_treated_npi_tag
  ) u
  WHERE npi IS NOT NULL AND patient_id IS NOT NULL
  GROUP BY npi
)
SELECT
  u.npi,
  COALESCE(dx.pts_dx, 0)                AS MPSII_All_Diagnosed,
  COALESCE(tx1.pts_all_dx_tx, 0)        AS MPSII_Diagnosed_Elaprase_Treated,
  COALESCE(tx2.pts_elaprase_treated, 0) AS MPSII_Elaprase_Treated,
  COALESCE(tot.pts_unique_all, 0)       AS All_patients_count
FROM HCP_Universe u
LEFT JOIN dx  ON u.npi = dx.npi
LEFT JOIN tx1 ON u.npi = tx1.npi
LEFT JOIN tx2 ON u.npi = tx2.npi
LEFT JOIN tot ON u.npi = tot.npi;


In [0]:
WITH joined AS (
  SELECT
    a.npi,
    CONCAT(c.first_name, ' ', c.last_name) AS hcp_name,
    c.primary_specialty,
    e.npi_num__v                            AS hco_npi,
    e.corporate_name__v                     AS hco_name,
    f.postal_code_cda__v                    AS hco_zip,
    g.name as hco_type_info,
    h.territory_name as hco_territory,
    a.mpsII_all_diagnosed,
    a.MPSII_Diagnosed_Elaprase_Treated,
    a.MPSII_Elaprase_Treated,
    a.All_patients_count,
    ROW_NUMBER() OVER (
      PARTITION BY a.npi
      ORDER BY d.modified_date__v DESC NULLS LAST,
               d.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM metric_numbers a
  LEFT JOIN com_edp_prd.com_raw.vod_hcp b
    ON a.npi = b.npi_num__v
  LEFT JOIN com_edp_prd.com_raw.kom_providers c
    ON a.npi = c.npi AND c.provider_type = 'INDIVIDUAL'
  LEFT JOIN com_edp_prd.com_raw.vod_parenthco d
    ON b.vid__v = d.entity_vid__v
   AND d.HIERARCHY_TYPE__V   = 'HCP_HCO'
   AND d.PARENT_HCO_STATUS__V = 'A'
   AND d.RELATIONSHIP_TYPE__V = '7356'
  LEFT JOIN com_edp_prd.com_raw.vod_hco e
    ON d.parent_hco_vid__v = e.vid__v
  LEFT JOIN com_edp_prd.com_raw.vod_address f
    ON d.parent_hco_vid__v = f.entity_vid__v
   AND f.entity_type__v = 'HCO'
   left join com_edp_prd.com_raw.vod_references g 
   on e.hco_type__v = g.code and g.reference_type = 'HCOType'
   left join com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping as h
   on f.postal_code_cda__v = h.zipcode
)
SELECT
  npi,
  hcp_name,
  primary_specialty,
  hco_npi,
  hco_name,
  hco_zip,
  hco_type_info,
  hco_territory,
  mpsII_all_diagnosed,
  MPSII_Diagnosed_Elaprase_Treated,
  MPSII_Elaprase_Treated,
  All_patients_count
FROM joined
WHERE rn = 1

